In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- base_pattern_iloc ---
FIX_BASE_PATTERN_ILOC_DS = SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.]}),y=pd.DataFrame({"y":[0,1,0,1,0]}),w=pd.DataFrame({"w":[1.,1.,1.,1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),filter=lambda mask: SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.]}),y=pd.DataFrame({"y":[0,1]}),w=pd.DataFrame({"w":[1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),save=lambda p:None),save=lambda p:None)
FIX_BASE_PATTERN_ILOC_TRAIN_IDX = [0, 1, 2]
FIX_BASE_PATTERN_ILOC_VALID_IDX = [0, 1, 2]

# --- base_pattern_pred_concat ---
FIX_BASE_PATTERN_PRED_CONCAT_DS = SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.]}),y=pd.DataFrame({"y":[0,1,0,1,0]}),w=pd.DataFrame({"w":[1.,1.,1.,1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.,3.,4.,5.],"x2":[4.,5.,6.,7.,8.],"y":[0,1,0,1,0],"w":[1.,1.,1.,1.,1.]}),filter=lambda mask: SimpleNamespace(X=pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.]}),y=pd.DataFrame({"y":[0,1]}),w=pd.DataFrame({"w":[1.,1.]}),x_columns=["x1","x2"],y_columns=["y"],w_columns=["w"],to_pandas=lambda:pd.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),to_polars=lambda:pl.DataFrame({"x1":[1.,2.],"x2":[4.,5.],"y":[0,1],"w":[1.,1.]}),save=lambda p:None),save=lambda p:None)
FIX_BASE_PATTERN_PRED_CONCAT_PRED = np.array([0.3, 0.7, 0.4, 0.8, 0.5])
FIX_BASE_PATTERN_PRED_CONCAT_VALID_IDX = [0, 1, 2, 3, 4]  # matches len(pred)

_pred_dfs = []

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_base_pattern_iloc(ds, train_idx, valid_idx):
    train_X = ds.X.iloc[train_idx]
    train_y = ds.y.iloc[train_idx].to_numpy().reshape(-1)
    train_w = ds.w.iloc[train_idx].to_numpy().reshape(-1)
    valid_X = ds.X.iloc[valid_idx]
    valid_y = ds.y.iloc[valid_idx].to_numpy().reshape(-1)
    valid_w = ds.w.iloc[valid_idx].to_numpy().reshape(-1)
    return valid_w

def before_base_pattern_pred_concat(ds, pred, valid_idx):
    _pred_dfs.append(
        pd.DataFrame(
            {"index": ds.y.index[valid_idx], "pred": pred.reshape(-1)}
        ).set_index("index")
    )
    pred_df = pd.concat(_pred_dfs, axis=0)
    base_df = pd.merge(
        ds.y.rename(columns={ds.y_columns[0]: "y"}),
        ds.w.rename(columns={ds.w_columns[0]: "w"}),
        left_index=True,
        right_index=True,
    )
    output_df = pd.merge(base_df, pred_df, left_index=True, right_index=True)
    return output_df

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_base_pattern_iloc(ds, train_idx, valid_idx):
    train_X = ds.X.take(train_idx)
    train_y = ds.y.take(train_idx).to_numpy().reshape(-1)
    train_w = ds.w.take(train_idx).to_numpy().reshape(-1)
    valid_X = ds.X.take(valid_idx)
    valid_y = ds.y.take(valid_idx).to_numpy().reshape(-1)
    valid_w = ds.w.take(valid_idx).to_numpy().reshape(-1)
    return valid_w

def gen_base_pattern_pred_concat(ds, pred, valid_idx):
    _pred_dfs.append(
        pl.DataFrame(
            {"index": ds.y.index[valid_idx], "pred": pred.reshape(-1)}
        )
    )
    pred_df = pl.concat(_pred_dfs, how="vertical")
    base_df = pl.DataFrame(
    {
        "index": ds.y.index,
        "y": ds.y.to_numpy(),
        "w": ds.w.to_numpy(),
    }
    )
    output_df = base_df.join(pred_df, on="index", how="inner")
    return output_df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: base_pattern_iloc ===

import sys

def _capture_return_locals(func, *args):
    captured = {}
    target_code = func.__code__
    def _trace(call_frame, event, arg):
        if event == "return" and call_frame.f_code is target_code:
            captured.update(call_frame.f_locals)
        return _trace
    old_trace = sys.gettrace()
    sys.settrace(_trace)
    try:
        result = func(*args)
    finally:
        sys.settrace(old_trace)
    return result, captured

def _values_equal(left, right):
    left_frame = _to_pl(left)
    right_frame = _to_pl(right)
    if left_frame is not None or right_frame is not None:
        if left_frame is None or right_frame is None or set(left_frame.columns) != set(right_frame.columns):
            return False
        try:
            pl_assert_frame_equal(
                left_frame.select(right_frame.columns),
                right_frame,
                check_dtypes=False,
                check_row_order=True,
            )
            return True
        except Exception:
            return False
    return np.array_equal(np.asarray(left), np.asarray(right))

def _compare_selected_locals(before_locals, gen_locals, layer):
    fields = ['train_X', 'train_y', 'train_w', 'valid_X', 'valid_y', 'valid_w']
    missing = [name for name in fields if name not in before_locals or name not in gen_locals]
    mismatched = [
        name for name in fields
        if name not in missing and not _values_equal(before_locals[name], gen_locals[name])
    ]
    if not missing and not mismatched:
        print(f"✅ {layer} base_pattern_iloc all positional selections: MATCH")
    else:
        print(f"❌ {layer} base_pattern_iloc all positional selections: MISMATCH — missing={missing}, mismatched={mismatched}")

try:
    _r = gen_base_pattern_iloc(FIX_BASE_PATTERN_ILOC_DS, FIX_BASE_PATTERN_ILOC_TRAIN_IDX, FIX_BASE_PATTERN_ILOC_VALID_IDX)
    print("✅ L1 smoke gen_base_pattern_iloc: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_base_pattern_iloc: {type(_e).__name__}: {_e}")

try:
    _rb = before_base_pattern_iloc(FIX_BASE_PATTERN_ILOC_DS, FIX_BASE_PATTERN_ILOC_TRAIN_IDX, FIX_BASE_PATTERN_ILOC_VALID_IDX)
    print("✅ L1 smoke before_base_pattern_iloc: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_base_pattern_iloc: {type(_e).__name__}: {_e}")

try:
    _, _before_locals = _capture_return_locals(before_base_pattern_iloc, FIX_BASE_PATTERN_ILOC_DS, FIX_BASE_PATTERN_ILOC_TRAIN_IDX, FIX_BASE_PATTERN_ILOC_VALID_IDX)
    _, _gen_locals = _capture_return_locals(gen_base_pattern_iloc, FIX_BASE_PATTERN_ILOC_DS, FIX_BASE_PATTERN_ILOC_TRAIN_IDX, FIX_BASE_PATTERN_ILOC_VALID_IDX)
    _compare_selected_locals(_before_locals, _gen_locals, "L2 equivalence")
except Exception as _e:
    print(f"❌ L2 equivalence base_pattern_iloc: setup error — {type(_e).__name__}: {_e}")

try:
    _, _before_locals = _capture_return_locals(before_base_pattern_iloc, FIX_BASE_PATTERN_ILOC_DS, [], [])
    _, _gen_locals = _capture_return_locals(gen_base_pattern_iloc, FIX_BASE_PATTERN_ILOC_DS, [], [])
    _compare_selected_locals(_before_locals, _gen_locals, "L3 edge")
except Exception as _e:
    print(f"❌ L3 edge base_pattern_iloc empty indices: {type(_e).__name__}: {_e}")
